In [1]:
from collections import defaultdict, Counter
import math

class KneserNeyLanguageModel:

    def __init__(self, n=5, discount=0.75):
        self.n = n
        self.discount = discount

        # n-gram counts
        self.ngram_counts = [defaultdict(int) for _ in range(n)]

        # context counts
        self.context_counts = [defaultdict(int) for _ in range(n)]

        # continuation statistics
        self.continuation = defaultdict(set)

        # vocabulary
        self.vocab = set()

    # -----------------------------
    # Tokenization
    # -----------------------------
    def tokenize(self, sentence):
        return sentence.lower().split()

    # -----------------------------
    # Train Model
    # -----------------------------
    def train(self, corpus):

        for sentence in corpus:

            tokens = ["<s>"]*(self.n-1)
            tokens += self.tokenize(sentence)
            tokens.append("</s>")

            self.vocab.update(tokens)

            length = len(tokens)

            for k in range(1, self.n+1):

                for i in range(length-k+1):

                    gram = tuple(tokens[i:i+k])

                    self.ngram_counts[k-1][gram] += 1

                    if k > 1:

                        context = gram[:-1]

                        self.context_counts[k-1][context] += 1

                        self.continuation[gram[-1]].add(context)

    # -----------------------------
    # Continuation Probability
    # -----------------------------
    def continuation_probability(self, word):

        numerator = len(self.continuation[word])

        denominator = len(self.context_counts[1])

        if denominator == 0:
            return 1 / len(self.vocab)

        return numerator / denominator

    # -----------------------------
    # Recursive Kneser Ney
    # -----------------------------
    def probability(self, history, word):

        history = tuple(history)

        if len(history) > self.n-1:
            history = history[-(self.n-1):]

        return self._recursive(history, word)

    def _recursive(self, history, word):

        order = len(history)+1

        # Base case (unigram)
        if order == 1:

            return self.continuation_probability(word)

        ngram = history + (word,)

        count_ngram = self.ngram_counts[order-1].get(ngram,0)

        count_history = self.context_counts[order-1].get(history,0)

        if count_history == 0:

            return self._recursive(history[1:],word)

        discounted = max(count_ngram-self.discount,0)/count_history

        unique_follow = 0

        for gram in self.ngram_counts[order-1]:

            if gram[:-1]==history:

                unique_follow += 1

        lambda_weight = (self.discount*unique_follow)/count_history

        lower_prob = self._recursive(history[1:],word)

        return discounted + lambda_weight*lower_prob

    # -----------------------------
    # Predict Next Words
    # -----------------------------
    def predict(self, sentence, top_k=5):

        tokens = self.tokenize(sentence)

        history = tokens[-(self.n-1):]

        scores = {}

        for word in self.vocab:

            if word=="<s>":
                continue

            scores[word]=self.probability(history,word)

        prediction = sorted(scores.items(),
                            key=lambda x:x[1],
                            reverse=True)

        return prediction[:top_k]

    # -----------------------------
    # Add New Trending Words
    # -----------------------------
    def update(self,new_sentence):

        self.train([new_sentence])
